# WATCHDOG Register Map Dokumentation

Dieses Notebook dokumentiert die Datei `watchdog/register_map.py`. Die Datei definiert alle Modbus-Register, die vom WATCHDOG-System zyklisch ausgelesen werden.

## Zweck

Die `REGISTER_MAP` dient als zentrale Konfigurations- und Abstraktionsschicht zwischen dem Modbus-Client und den physikalischen Datenpunkten der Wärmepumpe bzw. des Kaltwassersatzes.

Vorteile:
- Zentrale Definition aller Register
- Keine Hardcodierung im Modbus-Client
- Einfache Erweiterbarkeit
- Klare Trennung von Kommunikationslogik und Fachlogik

In [ ]:
from watchdog.register_map import REGISTER_MAP

REGISTER_MAP

## Aufbau eines Registereintrags

Jedes Register folgt derselben Struktur:

```python
{
    "address": 244,
    "count": 1,
    "type": "holding",
    "scale": 1,
    "unit": "°C",
    "description": "Vorlauftemperatur"
}
```

### Felder
- `address` → Modbus-Adresse
- `count` → Anzahl Register
- `type` → Holding- oder Input-Register
- `scale` → Skalierungsfaktor
- `unit` → Physikalische Einheit
- `description` → Lesbare Bezeichnung

## Enthaltene Messwerte

| Name | Adresse | Einheit | Beschreibung |
|------|----------|----------|--------------|
| user_outlet_temperature | 244 | °C | Vorlauftemperatur |
| user_inlet_temperature | 245 | °C | Rücklauftemperatur |
| outdoor_temperature | 247 | °C | Außentemperatur |
| compressor_speed | 248 | rpm | Verdichterdrehzahl |
| compressor1_current | 249 | A | Stromaufnahme Verdichter 1 |
| operating_mode | 0 | - | Betriebsart |
| pump_running_speed | 137 | % | Pumpendrehzahl |
| setpoint | 1 | °C | Sollwert |

## Datenfluss

```text
REGISTER_MAP
     │
     ▼
modbus_client.read_all_registers()
     │
     ▼
Messwert-Dictionary
     │
     ▼
database.insert_measurements()
     │
     ▼
SQLite Datenbank
```

## ADR-005: Zentrale Registerdefinition

### Status
Accepted

### Entscheidung
Alle Modbus-Register werden zentral in `register_map.py` definiert.

### Vorteile
- Einfache Wartung
- Übersichtliche Dokumentation
- Kein doppelter Code
- Erweiterungen ohne Änderungen am Modbus-Client

### Nachteile
- Wachsende Datei bei vielen Datenpunkten
- Keine automatische Validierung der Adressen

### Alternativen
- YAML-Datei
- JSON-Datei
- Datenbankgestützte Registerdefinition

### Zukunft
- Validierung der Registerdefinitionen
- Gruppen für Temperatur-, Druck- und Statuswerte
- Automatische Generierung von Dokumentation aus der Register Map

## Verbesserungsvorschläge

1. Unterstützung mehrerer Registerblöcke (`count > 1`)
2. Datentypen (INT16, UINT16, FLOAT32) ergänzen
3. Automatische Plausibilitätsprüfung
4. Generierung einer technischen Registerliste für Serviceeinsätze
5. Verknüpfung mit Herstellerdokumentationen